In [ ]:
import json
import os

# Path to your kaggle.json
kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")

# Load credentials
with open(kaggle_json_path) as f:
    creds = json.load(f)

# Set environment variables
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']

print("Kaggle credentials loaded successfully!")
!kaggle datasets download -d navoneel/brain-mri-images-for-brain-tumor-detection -p ./dataset --unzip


# List files in the dataset folder
print(os.listdir('./dataset'))



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection")

print("Path to dataset files:", path)

In [ ]:

DATASET_PATH = path  

print("Classes found in dataset folder:", os.listdir(DATASET_PATH))
print("Number of 'yes' images:", len(os.listdir(os.path.join(DATASET_PATH, 'yes'))))
print("Number of 'no' images:", len(os.listdir(os.path.join(DATASET_PATH, 'no'))))

print("\nTraining images (VGG):", train_vgg.samples)
print("Validation images (VGG):", val_vgg.samples)
print("Test images (VGG):", test_vgg.samples)
pribt("total images:", train_vgg.samples + test_vgg.samples + val_vgg.samples)

In [ ]:

import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet import preprocess_input as resnet_preprocess
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model


RANDOM_SEED = 123
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 1  # binary classification
EPOCHS = 20



# Use validation_split to split within yes/no subfolders
train_val_datagen_vgg = ImageDataGenerator(
    preprocessing_function=vgg_preprocess,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    brightness_range=[0.5, 1.5],
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2  # 80% train, 20% val
)

base_datagen_vgg = ImageDataGenerator(
    preprocessing_function=vgg_preprocess,
    validation_split=0.1  # 10% test split
)

train_val_datagen_resnet = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    brightness_range=[0.5, 1.5],
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2
)

base_datagen_resnet = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    validation_split=0.1
)

# -----------------------------
# VGG16 SPLITS
# -----------------------------
train_vgg = train_val_datagen_vgg.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['no', 'yes'],     # ✅ explicitly use only these folders
    subset='training',
    seed=RANDOM_SEED
)

val_vgg = train_val_datagen_vgg.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['no', 'yes'],
    subset='validation',
    seed=RANDOM_SEED
)

test_vgg = base_datagen_vgg.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['no', 'yes'],
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)

# -----------------------------
# RESNET50 SPLITS
# -----------------------------
train_resnet = train_val_datagen_resnet.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['no', 'yes'],
    subset='training',
    seed=RANDOM_SEED
)

val_resnet = train_val_datagen_resnet.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['no', 'yes'],
    subset='validation',
    seed=RANDOM_SEED
)

test_resnet = base_datagen_resnet.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['no', 'yes'],
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)


# -----------------------------
# MODEL BUILDERS
# -----------------------------
def build_vgg16(input_shape=(224,224,3)):
    base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy',
                  optimizer=RMSprop(learning_rate=1e-4),
                  metrics=['accuracy'])
    return model

def build_resnet50(input_shape=(224,224,3)):
    base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)
    model = models.Model(inputs=base.input, outputs=outputs)
    model.compile(loss='binary_crossentropy',
                  optimizer=RMSprop(learning_rate=1e-4),
                  metrics=['accuracy'])
    return model


In [ ]:
es = EarlyStopping(monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True)
checkpoint_vgg = ModelCheckpoint('vgg16_best.h5', save_best_only=True, monitor='val_accuracy', mode='max')
checkpoint_resnet = ModelCheckpoint('resnet50_best.h5', save_best_only=True, monitor='val_accuracy', mode='max')

# -----------------------------
# TRAIN MODELS
# -----------------------------
vgg_model = build_vgg16()
print("\nTraining VGG16...\n")
history_vgg = vgg_model.fit(
    train_vgg,
    validation_data=val_vgg,
    epochs=EPOCHS,
    callbacks=[checkpoint_vgg],
    verbose=1
)


# -----------------------------
# EVALUATE ON TEST SET
# -----------------------------
print("\nEvaluating VGG16 on Test Set...")
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(test_vgg)
print(f"VGG16 Test Accuracy: {vgg_test_acc:.4f}")

# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np

# # 1. Get class names
# class_indices = train_vgg.class_indices
# classes = list(class_indices.keys())
# print("Classes:", classes)

# # 2. Predict on validation set
# val_steps = val_vgg.samples // val_vgg.batch_size
# y_pred_probs = vgg_model.predict(val_vgg, steps=val_steps+1)
# y_pred = np.argmax(y_pred_probs, axis=1)

# # 3. True labels
# y_true = val_vgg.classes[:len(y_pred)]  # match length

# # 4. Confusion matrix
# cm = confusion_matrix(y_true, y_pred)
# plt.figure(figsize=(6,5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
# plt.xlabel("Predicted")
# plt.ylabel("Actual")
# plt.title("Confusion Matrix")
# plt.show()

# # 5. Classification report
# report = classification_report(y_true, y_pred, target_names=classes)
# print("Classification Report:\n", report)

# # 6. Other metrics
# accuracy = accuracy_score(y_true, y_pred)
# precision_macro = precision_score(y_true, y_pred, average='macro')
# recall_macro = recall_score(y_true, y_pred, average='macro')
# f1_macro = f1_score(y_true, y_pred, average='macro')

# precision_micro = precision_score(y_true, y_pred, average='micro')
# recall_micro = recall_score(y_true, y_pred, average='micro')
# f1_micro = f1_score(y_true, y_pred, average='micro')

# print(f"Accuracy: {accuracy:.4f}")
# print(f"Precision (Macro): {precision_macro:.4f}, Recall (Macro): {recall_macro:.4f}, F1 (Macro): {f1_macro:.4f}")
# print(f"Precision (Micro): {precision_micro:.4f}, Recall (Micro): {recall_micro:.4f}, F1 (Micro): {f1_micro:.4f}")



In [ ]:
resnet_model = build_resnet50()
print("\nTraining ResNet50...\n")
history_resnet = resnet_model.fit(
    train_resnet,
    validation_data=val_resnet,
    epochs=EPOCHS,
    callbacks=[es, checkpoint_resnet],
    verbose=1
)

print("\nEvaluating ResNet50 on Test Set...")
resnet_test_loss, resnet_test_acc = resnet_model.evaluate(test_resnet)
print(f"ResNet50 Test Accuracy: {resnet_test_acc:.4f}")



In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing import image
import tensorflow as tf

# Input
inputs = layers.Input(shape=(224, 224, 3), name="img")

# VGG16
vgg = VGG16(input_tensor=inputs, weights='imagenet', include_top=False)
for layer in vgg.layers:
    layer.trainable = False
vgg_out = layers.GlobalAveragePooling2D()(vgg.output)

# ResNet50
resnet = ResNet50(input_tensor=inputs, weights='imagenet', include_top=False)
for layer in resnet.layers:
    layer.trainable = False
resnet_out = layers.GlobalAveragePooling2D()(resnet.output)

# Merge
merged = layers.Concatenate()([vgg_out, resnet_out])
x = layers.Dense(256, activation="relu")(merged)
x = layers.Dropout(0.5)(x)  # Optional dropout
prediction = layers.Dense(2, activation='softmax')(x)

# Model
model = Model(inputs=inputs, outputs=prediction)
model.compile(loss="categorical_crossentropy", optimizer='adam', metrics=['accuracy'])
model.summary()

# Data generators
train_datagen = image.ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = image.ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    classes=['no', 'yes']
)
validation_generator = test_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    classes=['no', 'yes']
)

#


In [ ]:
 #Training
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=20,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size
)

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
validation_generator = test_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    classes=['no', 'yes'],
    shuffle=False
)
# 1. Get class names
classes = list(train_generator.class_indices.keys())

# 2. Predict on validation set
val_steps = validation_generator.samples // validation_generator.batch_size
y_pred_probs = model.predict(validation_generator, steps=val_steps+1)
y_pred = np.argmax(y_pred_probs, axis=1)

# 3. True labels
y_true = validation_generator.classes[:len(y_pred)]

# 4. Classification report dictionary
report_dict = classification_report(y_true, y_pred, target_names=classes, output_dict=True)

# 5. Create table
metrics_table = pd.DataFrame(report_dict).transpose()

# 6. Add overall accuracy row manually
metrics_table.loc['overall_accuracy'] = [accuracy_score(y_true, y_pred), np.nan, np.nan, np.nan]

# 7. Round values for readability
metrics_table = metrics_table.round(4)

# 8. Display
print(metrics_table)


In [ ]:
loss, acc = model.evaluate(validation_generator)
print("Validation accuracy (evaluate):", acc)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Get class names
class_indices = train_generator.class_indices
classes = list(class_indices.keys())
print("Classes:", classes)

# 2. Predict on validation set
val_steps = validation_generator.samples // validation_generator.batch_size
y_pred_probs = model.predict(validation_generator, steps=val_steps+1)
y_pred = np.argmax(y_pred_probs, axis=1)

# 3. True labels
y_true = validation_generator.classes[:len(y_pred)]  # match length

# 4. Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# 5. Classification report
report = classification_report(y_true, y_pred, target_names=classes)
print("Classification Report:\n", report)

# 6. Other metrics
accuracy = accuracy_score(y_true, y_pred)
precision_macro = precision_score(y_true, y_pred, average='macro')
recall_macro = recall_score(y_true, y_pred, average='macro')
f1_macro = f1_score(y_true, y_pred, average='macro')

precision_micro = precision_score(y_true, y_pred, average='micro')
recall_micro = recall_score(y_true, y_pred, average='micro')
f1_micro = f1_score(y_true, y_pred, average='micro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (Macro): {precision_macro:.4f}, Recall (Macro): {recall_macro:.4f}, F1 (Macro): {f1_macro:.4f}")
print(f"Precision (Micro): {precision_micro:.4f}, Recall (Micro): {recall_micro:.4f}, F1 (Micro): {f1_micro:.4f}")
